
# BingePlay Streaming Analytics
Setting up the DB connection. Make sure your local MySQL server is running.


In [22]:
# 1. Install MySQL Server
!apt-get update
!apt-get install mysql-server -y > /dev/null

# 2. Start the MySQL service
!service mysql start

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld
   ...done.


In [23]:
# 3. Create database and load the SQL file
!mysql -u root -proot -e "CREATE DATABASE IF NOT EXISTS bingeplay;"
!mysql -u root -proot bingeplay < "/content/Advanced SQL Data set for MP 4.sql"
print("Database 'bingeplay' created and data loaded successfully.")

mysql: [Warning] Using a password on the command line interface can be insecure.
mysql: [Warning] Using a password on the command line interface can be insecure.
tbl	row_count
users	3000
subscriptions	4497
shows	100
watch_sessions	100351
ratings	5000
null_user_sessions
2
Database 'bingeplay' created and data loaded successfully.


In [24]:
import pandas as pd
from sqlalchemy import create_engine

# Now we connect to the local server we just started
USER = 'root'
PASSWORD = 'root'
HOST = '127.0.0.1'
DB_NAME = 'bingeplay'

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")
print("MySQL engine ready and connected to local server.")

MySQL engine ready and connected to local server.


## Q1 — Active revenue
**Active subscriptions:** 2340  
**Total monthly revenue:** ₹7,84,260  

gotta remember end_date is null here, otherwise it drops all the still-open subs.

In [25]:
query = """
select count(*) as active_subs,
       sum(monthly_price_inr) as monthly_revenue
from subscriptions
where status = 'active'
  and (end_date is null or end_date > '2024-06-30');
"""
pd.read_sql(query, engine)

,active_subs,monthly_revenue
0,2340,784260.0


## Q2 — Signup momentum
group by month for 2024 signups.

In [26]:
query = """
select month(signup_date) as signup_month,
       count(*) as new_users
from users
group by month(signup_date)
order by signup_month;
"""
pd.read_sql(query, engine)

,signup_month,new_users
0,1,350
1,2,400
2,3,500
3,4,550
4,5,600
5,6,600



## Q3 — Device analytics
Quick check on total sessions, mins, and completion rate per device type.

In [27]:
query = """
select device_type,
       count(*) as total_sessions,
       sum(watch_minutes) as total_minutes,
       round(avg(watch_minutes), 2) as avg_minutes,
       round(sum(completed) / count(*) * 100, 2) as completion_rate_pct
from watch_sessions
where user_id is not null
group by device_type;
"""
pd.read_sql(query, engine)

,device_type,total_sessions,total_minutes,avg_minutes,completion_rate_pct
0,Mobile,50172,1504355.0,29.98,60.24
1,TV,27981,840595.0,30.04,59.98
2,Laptop,15105,453434.0,30.02,60.51
3,Tablet,7091,210733.0,29.72,59.79



## Q4 — Rating distribution
Using a subquery here instead of hardcoding the 5k total rows just in case more data comes in later.


In [28]:
query = """
select stars,
       count(*) as cnt,
       round(count(*) * 100.0 / (select count(*) from ratings), 2) as pct
from ratings
group by stars
order by stars;
"""
pd.read_sql(query, engine)

,stars,cnt,pct
0,1,234,4.68
1,2,352,7.04
2,3,847,16.94
3,4,1781,35.62
4,5,1786,35.72


In [29]:
query2 = """
select round(sum(case when stars in (4,5) then 1 else 0 end) / count(*) * 100, 2) as pct_4_5_stars
from ratings;
"""
pd.read_sql(query2, engine)

,pct_4_5_stars
0,71.34



## Q5 — Originals vs acquired
Originals seem to be carrying the average rating.


In [30]:
query = """
select is_original,
       count(*) as num_shows,
       round(avg(imdb_rating), 2) as avg_rating,
       round(avg(release_year), 2) as avg_year
from shows
group by is_original;
"""
pd.read_sql(query, engine)

,is_original,num_shows,avg_rating,avg_year
0,0,70,6.63,2020.73
1,1,30,7.92,2020.37


## Q6 — Binge day detection

Total binge days in Q2. Grouping by user/show/date so different shows on the same day don't mess up the count.


In [31]:
query = """
select count(*) as total_binge_days
from (
    select user_id, show_id, session_date
    from watch_sessions
    where session_date between '2024-04-01' and '2024-06-30'
    group by user_id, show_id, session_date
    having count(*) >= 5
) t;
"""
pd.read_sql(query, engine)

,total_binge_days
0,414


In [32]:
query2 = """
select user_id, count(*) as binge_days
from (
    select user_id, show_id, session_date
    from watch_sessions
    where session_date between '2024-04-01' and '2024-06-30'
    group by user_id, show_id, session_date
    having count(*) >= 5
) tmp
group by user_id
order by binge_days desc
limit 5;
"""
pd.read_sql(query2, engine)

,user_id,binge_days
0,U02956,8
1,U02292,6
2,U02334,5
3,U00118,5
4,U02583,4


## Q7 — signups who never watched
null trap handling: left join to find missing sessions instead of NOT IN, which breaks if there are nulls in the subquery.

In [33]:
query = """
select count(*) as q1_signups
from users
where signup_date between '2024-01-01' and '2024-03-31';
"""
pd.read_sql(query, engine)

,q1_signups
0,1250


In [34]:
query2 = """
select count(u.user_id) as never_watched
from users u
left join watch_sessions ws on u.user_id = ws.user_id
where u.signup_date between '2024-01-01' and '2024-03-31'
  and ws.session_id is null;
"""
pd.read_sql(query2, engine)

,never_watched
0,226


## Q8 — Over-paying Premium/Family users
users paying for top tiers but only ever watching basic shows.

In [35]:
query = """
select count(*) as overpaying_users
from (
    select user_id, plan
    from subscriptions
    where status = 'active'
      and (end_date is null or end_date > '2024-06-30')
) cur
where cur.plan in ('Premium', 'Family')
  and exists (
      select 1 from watch_sessions ws where ws.user_id = cur.user_id
  )
  and not exists (
      select 1
      from watch_sessions ws
      join shows sh on sh.show_id = ws.show_id
      where ws.user_id = cur.user_id
        and sh.min_plan in ('Premium', 'Family')
  );
"""
pd.read_sql(query, engine)

,overpaying_users
0,7



## Q9 — Upgrade success cohort
Checking basic users from Jan who eventually bumped up to Premium/Family.


In [36]:
query = """
with ranked_subs as (
    select s.*,
           row_number() over (partition by s.user_id order by s.start_date) as rn
    from subscriptions s
    join users u on u.user_id = s.user_id
    where month(u.signup_date) = 1 and year(u.signup_date) = 2024
),
first_sub as (
    select user_id, plan as first_plan, start_date as first_start
    from ranked_subs
    where rn = 1
),
upgrades as (
    select r.user_id, min(r.start_date) as upgrade_date
    from ranked_subs r
    join first_sub f on f.user_id = r.user_id
    where r.plan in ('Premium', 'Family') and r.start_date > f.first_start
    group by r.user_id
),
active_users as (
    select user_id
    from subscriptions
    where status = 'active' and (end_date is null or end_date > '2024-06-30')
)
select count(*) as upgraders,
       round(avg(datediff(u2.upgrade_date, f.first_start)), 2) as avg_days_to_upgrade
from first_sub f
join upgrades u2 on u2.user_id = f.user_id
join active_users ca on ca.user_id = f.user_id
where f.first_plan = 'Basic';
"""
pd.read_sql(query, engine)

,upgraders,avg_days_to_upgrade
0,55,61.25


## Q10 — Cliffhanger comebacks
users coming back to incomplete shows within 1 to 7 days.

In [37]:
query = """
select count(*) as total_comebacks
from (
    select distinct a.user_id, a.show_id, a.session_date
    from watch_sessions a
    join watch_sessions b
      on a.user_id = b.user_id
      and a.show_id = b.show_id
      and a.completed = 0
      and b.session_date > a.session_date
      and datediff(b.session_date, a.session_date) between 1 and 7
) t;
"""
pd.read_sql(query, engine)

,total_comebacks
0,4345


In [38]:
query2 = """
select sh.show_id, sh.title, count(*) as comeback_count
from (
    select distinct a.user_id, a.show_id, a.session_date
    from watch_sessions a
    join watch_sessions b
      on a.user_id = b.user_id
      and a.show_id = b.show_id
      and a.completed = 0
      and b.session_date > a.session_date
      and datediff(b.session_date, a.session_date) between 1 and 7
) t
join shows sh on sh.show_id = t.show_id
group by sh.show_id, sh.title
order by comeback_count desc
limit 5;
"""
pd.read_sql(query2, engine)

,show_id,title,comeback_count
0,S088,Rayalaseema Raga,64
1,S019,Founder Diaries,61
2,S005,Show Special 27,61
3,S024,Show Special 18,59
4,S065,Show Special 21,58



## Q11 — Consecutive-week engagement
Using the wk start dates to avoid that annoying yearweek rollover bug.


In [44]:
query = """
-- Tracking users who stayed active for 4+ weeks straight
-- Using a self-join approach here
with wks as (
    select distinct user_id,
           yearweek(session_date, 3) as wk_id
    from watch_sessions
    where user_id is not null
)
select count(distinct w1.user_id) as users_with_streak
from wks w1
join wks w2 on w1.user_id = w2.user_id and w2.wk_id = w1.wk_id + 1
join wks w3 on w1.user_id = w3.user_id and w3.wk_id = w1.wk_id + 2
join wks w4 on w1.user_id = w4.user_id and w4.wk_id = w1.wk_id + 3;
"""
pd.read_sql(query, engine)

,users_with_streak
0,1675


In [47]:
query2 = """
-- Finding the actual heavy hitters (top streaks)
-- Sticking with the group-by logic from the previous question
with user_activity as (
    select user_id,
           wk as week_val,
           -- create a grouping key by subtracting a counter from the week number
           wk - row_number() over(partition by user_id order by wk) as streak_group
    from (select distinct user_id, yearweek(session_date, 3) as wk from watch_sessions) d
    where user_id is not null
),
streak_calc as (
    select user_id, count(*) as len
    from user_activity
    group by user_id, streak_group
)
select user_id, max(len) as max_streak
from streak_calc
group by 1
order by 2 desc
limit 5;
"""
pd.read_sql(query2, engine)

,user_id,max_streak
0,U01259,26
1,U01658,26
2,U00529,26
3,U01793,26
4,U00213,26



## Q12 — Churn signal detection
50% drop from May to June. Sticking this in a CTE to make the math easier to filter.


In [48]:
query = """
-- Churn check: who dropped off by 50%% or more between May and June?
select count(*) as churn_count
from (
    select user_id,
           sum(if(month(session_date)=5, watch_minutes, 0)) as m5,
           sum(if(month(session_date)=6, watch_minutes, 0)) as m6
    from watch_sessions
    where user_id is not null
    group by user_id
) activity
where m5 > 0
  and (m5 - m6) / m5 >= 0.5;
"""
pd.read_sql(query, engine)

,churn_count
0,521


In [43]:
query2 = """
with monthly as (
    select user_id,
        sum(case when month(session_date) = 5 then watch_minutes else 0 end) as may_mins,
        sum(case when month(session_date) = 6 then watch_minutes else 0 end) as jun_mins
    from watch_sessions
    where user_id is not null and month(session_date) in (5, 6)
    group by user_id
)
select m.user_id, u.name, m.may_mins, m.jun_mins,
       round((m.may_mins - m.jun_mins) / m.may_mins * 100, 2) as drop_pct
from monthly m
join users u on u.user_id = m.user_id
where m.may_mins > 0
  and (m.may_mins - m.jun_mins) / m.may_mins >= 0.5
order by drop_pct desc
limit 10;
"""
pd.read_sql(query2, engine)

,user_id,name,may_mins,jun_mins,drop_pct
0,U00023,Amit Mukherjee,43.0,0.0,100.0
1,U00166,Shaurya Malhotra,94.0,0.0,100.0
2,U00211,Ravi Menon,336.0,0.0,100.0
3,U00225,Kritika Patil,209.0,0.0,100.0
4,U00237,Shaurya Bansal,126.0,0.0,100.0
5,U00262,Suresh Roy,158.0,0.0,100.0
6,U00271,Krish Patel,227.0,0.0,100.0
7,U00282,Tara Kumar,47.0,0.0,100.0
8,U00289,Rohan Krishnan,18.0,0.0,100.0
9,U00292,Kavitha Kamath,594.0,0.0,100.0
